In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import utils, prompts

In [2]:
import os
import re
import json
import ast

import pandas as pd

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
gigachat_ak = os.getenv("GIGACHAT_AK")
persist_dir = '../db/chroma'

In [5]:
LLM = "GigaChat-2"

In [6]:
#EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

### Запрос пользователя для прогона по пайплайну

In [7]:
user_query = "Посоветуй ЖК от ПИК в Химках или Мытищах"

In [8]:
right_answer = "['Яуза парк', 'Ярославский', 'Ярославский квартал']"

### Подключение к векторной базе

In [9]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [10]:
client = chromadb.PersistentClient(path=persist_dir)
collections = client.list_collections()
collections

[Collection(name=chunk_700_qwen),
 Collection(name=chunk_300_se),
 Collection(name=chunk_500_qwen),
 Collection(name=chunk_300_qwen),
 Collection(name=chunk_700_se),
 Collection(name=chunk_500_se),
 Collection(name=chunk_900_se),
 Collection(name=chunk_900_qwen)]

In [11]:
collection_name = "chunk_700_se"

In [14]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

In [15]:
vectorstore = Chroma(
            persist_directory=persist_dir,
            embedding_function=embeddings,
            collection_name=collection_name,
        )

### Пример поиска без фильтрации

In [16]:
docs = vectorstore.similarity_search(user_query, k=5)

In [17]:
docs

[Document(id='beef1f0d-fef4-40ae-afcf-0d59b8d10aeb', metadata={'metro': 'Курьяново(D2)', 'proj': 'Квартал домашний', 'dist_to_metro': 0.5, 'h_class': 'Комфорт', 'distr': 'Марьино', 'dev': 'Самолет', 'n_fact': 0, 'num_resid': 3500, 's_plan': 875, 'parking': 'Подземный', 's_fact': 0, 'n_plan': 380}, page_content='москвы братеевская пойма природный заказник историческое курьяново квартал домашний место город природа становятся частью повседневной жизни'),
 Document(id='cc99e747-e8df-4ecc-a1b2-2b623f1ba2ef', metadata={'distr': 'Молжаниновский', 'dev': 'Самолет', 'n_fact': 350, 's_fact': 0, 'proj': 'Молжаниново', 'metro': 'Беломорская', 's_plan': 1000, 'num_resid': 20000, 'h_class': 'Комфорт', 'n_plan': 1000, 'dist_to_metro': 11.9, 'parking': 'Не предусмотрен'}, page_content='пассажирский парковки автомобили оставить многоуровневом плоскостном паркинге'),
 Document(id='fcc39474-73a0-4bf1-97c5-64beff95e887', metadata={'s_fact': 0, 'n_fact': 0, 'parking': 'Подземный', 'num_resid': 3500, 'dev'

### Выделение ключей из запроса

In [18]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_gigachat.chat_models import GigaChat
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser

In [19]:
json_parser = JsonOutputParser()

In [20]:
metadata_prompt = ChatPromptTemplate.from_messages([
    ("system", prompts.metadata_system),
    ("human", "{{question}}"),
], template_format="jinja2")

In [21]:
llm = GigaChat(
    credentials=gigachat_ak,
    model=LLM,  
    verify_ssl_certs=False 
)

In [22]:
extractor_chain = (
    RunnablePassthrough()
    | metadata_prompt 
    | llm 
    | json_parser
)

### Пример выделения метаданных

In [23]:
response = extractor_chain.invoke({"question": user_query})

In [24]:
response

{'dev': 'ПИК', 'distr': {'$in': ['Химки', 'Мытищи']}}

In [25]:
parsed_filter = utils.parse_filter(response)
parsed_filter

{'dev': 'ПИК', 'distr': {'$in': ['Химки', 'Мытищи']}}

In [26]:
normalized_filter = utils.normalize_chroma_filter(parsed_filter)
normalized_filter

{'$and': [{'dev': {'$eq': 'ПИК'}}, {'distr': {'$in': ['Химки', 'Мытищи']}}]}

### Augmented: обогащение контекстом

In [27]:
vectore_results = vectorstore.similarity_search(
    query=user_query,
    k=5,
    filter=normalized_filter,
)

In [28]:
test_context = utils.get_context(vectore_results)
test_context

'[START_BLOCK][START_META_BLOCK]Застройщик: ПИК\nНазвание ЖК: Ярославский\nКласс жилья: Комфорт\nРайон: Мытищи\nМетро: Медведково\nРасстояние до метро в км: 5.7\nКоличество жильцов: 8000\nПланируемое количество мест в школе: 2700\nДоступных мест в школе сейчас: 2700\nПланируемое количество мест в детсадах: 735\nДоступных мест в детсадах сейчас: 435\nТип парковки: Надземный\n[END_META_BLOCK]Ярославский: двумя санузлами разнообразными балконами квартиры сдаются готовой отделкой переехать сразу получения ключей\n[END_BLOCK]\n\n[START_BLOCK][START_META_BLOCK]Застройщик: ПИК\nНазвание ЖК: Яуза парк\nКласс жилья: Комфорт\nРайон: Мытищи\nМетро: Медведково\nРасстояние до метро в км: 9.0\nКоличество жильцов: 5500\nПланируемое количество мест в школе: 1170\nДоступных мест в школе сейчас: 0\nПланируемое количество мест в детсадах: 575\nДоступных мест в детсадах сейчас: 225\nТип парковки: Надземный\n[END_META_BLOCK]Яуза парк: яуза парк современный квартал подмосковных мытищах 5 минутах пешком одно

### Generation: создание ответа

In [30]:
base_prompt = ChatPromptTemplate.from_messages([
    ('system', prompts.system_prompt_2),
    ('human', prompts.human_prompt)
], template_format="jinja2")

In [35]:
def rag_with_filter(user_question: str) -> str:
    # 1. Получаем фильтр из вопроса
    raw_filter = extractor_chain.invoke({"question": user_question})
    print("DEBUG raw filter from LLM:", raw_filter, "type:", type(raw_filter))

    # 2. Парсим фильтр в словарь
    parsed_filter = utils.parse_filter(raw_filter)
    print("DEBUG parsed filter from LLM:", parsed_filter, "type:", type(parsed_filter))
            
    # 3. Нормализуем фильтр под требования Chroma
    normalized_filter = utils.normalize_chroma_filter(parsed_filter)
    print("DEBUG normalized filter:", normalized_filter, "type:", type(normalized_filter))

    # 4. Создаём ретривер
    if normalized_filter is not None:
        search_kwargs = {"k": 5, "filter": normalized_filter}
        filtered_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
        docs = filtered_retriever.invoke(user_question)
        print(f"[RAG] Поиск с фильтром {normalized_filter} → найдено: {len(docs)}")
    else:
        docs = vectorstore.as_retriever(k=5).invoke(user_question)
        print("[RAG] Фильтр не извлечён или пуст → поиск без фильтра")

    # 5. Если пусто — повторный поиск без фильтра
    if not docs:
        print("[RAG] Первый поиск вернул пустоту → повторный поиск без фильтра")
        docs = vectorstore.as_retriever(k=5).invoke(user_question)
        print(f"[RAG] Повторный поиск без фильтра → найдено: {len(docs)}")

    # 6. Если и после этого пусто — не отправляем в LLM
    if not docs:
        return "Не знаю"

    # 7. Формируем контекст
    context_str = utils.get_context(docs)

    # 8. Готовим промпт и вызываем LLM
    messages = base_prompt.format_messages(context=context_str, question=user_question)
    response = llm.invoke(messages)

    return response.content

In [36]:
answer = rag_with_filter(user_query)
print(answer)

DEBUG raw filter from LLM: {'dev': 'ПИК', 'distr': {'$in': ['Химки', 'Мытищи']}} type: <class 'dict'>
DEBUG parsed filter from LLM: {'dev': 'ПИК', 'distr': {'$in': ['Химки', 'Мытищи']}} type: <class 'dict'>
DEBUG normalized filter: {'$and': [{'dev': {'$eq': 'ПИК'}}, {'distr': {'$in': ['Химки', 'Мытищи']}}]} type: <class 'dict'>
[RAG] Поиск с фильтром {'$and': [{'dev': {'$eq': 'ПИК'}}, {'distr': {'$in': ['Химки', 'Мытищи']}}]} → найдено: 5
Из предоставленных данных подходят два ЖК от ПИК: "Ярославский" и "Яуза парк", оба находятся в Мытищах, а не в Химках.

1. **Ярославский**:
   - Район: Мытищи
   - Метро: Медведково
   - Расстояние до метро: 5.7 км
   - Количество жильцов: 8000
   - Тип парковки: надземный
   
2. **Яуза парк**:
   - Район: Мытищи
   - Метро: Медведково
   - Расстояние до метро: 9.0 км
   - Количество жильцов: 5500
   - Тип парковки: надземный

Ответ: Оба ЖК "Ярославский" и "Яуза парк" находятся в Мытищах.


### Оценка качества на тестовом наборе вопросов

In [37]:
questions = []

In [38]:
with open(r'../raw_data/questions_lite.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        data = dict(record)
        questions.append(data)

In [39]:
questions

[{'question': 'Какое плановое количество жителей указано для проекта «Восточное Бутово»?',
  'answer': ['30000']},
 {'question': 'Относится ли проект «2-й Иртышский» к типу «Комфорт плюс»?',
  'answer': ['Да']},
 {'question': 'В каком проекте плановая вместимость школ больше: «Ильинские луга» или «Восточное Бутово»?',
  'answer': ['Восточное Бутово']},
 {'question': 'Какой проект расположен дальше от метро: «Квартал Авиаторов» или «Остафьево»?',
  'answer': ['Квартал Авиаторов']},
 {'question': 'В каком проекте плановое количество жителей больше: «Пригород Лесное» или «Новое Видное»?',
  'answer': ['Пригород Лесное']}]

In [40]:
for question in questions:
    print(question)
    user_query = question['question']
    answer_rag = rag_with_filter(user_query)
    question['answer_rag'] = answer_rag

{'question': 'Какое плановое количество жителей указано для проекта «Восточное Бутово»?', 'answer': ['30000']}
DEBUG raw filter from LLM: {'proj': 'Восточное Бутово', 'num_resid': {'$exists': True}} type: <class 'dict'>
DEBUG parsed filter from LLM: {'proj': 'Восточное Бутово', 'num_resid': {'$exists': True}} type: <class 'dict'>
DEBUG normalized filter: {'proj': {'$eq': 'Восточное Бутово'}} type: <class 'dict'>
[RAG] Поиск с фильтром {'proj': {'$eq': 'Восточное Бутово'}} → найдено: 3
{'question': 'Относится ли проект «2-й Иртышский» к типу «Комфорт плюс»?', 'answer': ['Да']}
DEBUG raw filter from LLM: {'proj': '2-й Иртышский', 'h_class': 'Комфорт плюс'} type: <class 'dict'>
DEBUG parsed filter from LLM: {'proj': '2-й Иртышский', 'h_class': 'Комфорт плюс'} type: <class 'dict'>
DEBUG normalized filter: {'$and': [{'proj': {'$eq': '2-й Иртышский'}}, {'h_class': {'$eq': 'Комфорт плюс'}}]} type: <class 'dict'>
[RAG] Поиск с фильтром {'$and': [{'proj': {'$eq': '2-й Иртышский'}}, {'h_class': 

In [41]:
questions

[{'question': 'Какое плановое количество жителей указано для проекта «Восточное Бутово»?',
  'answer': ['30000'],
  'answer_rag': 'Ответ: Плановое количество жителей для проекта «Восточное Бутово» составляет 30000 человек.'},
 {'question': 'Относится ли проект «2-й Иртышский» к типу «Комфорт плюс»?',
  'answer': ['Да'],
  'answer_rag': 'Проект «2-й Иртышский» относится к типу «Комфорт плюс».\n\nОтвет: Да'},
 {'question': 'В каком проекте плановая вместимость школ больше: «Ильинские луга» или «Восточное Бутово»?',
  'answer': ['Восточное Бутово'],
  'answer_rag': 'Для того чтобы определить, в каком проекте плановая вместимость школ больше, сравним данные по каждому проекту:\n\n1. Проект "Ильинские луга":\n   - Планируемое количество мест в школе: 3000\n\n2. Проект "Восточное Бутово":\n   - Планируемое количество мест в школе: 5400\n\nИз этих данных видно, что плановая вместимость школ в проекте "Восточное Бутово" больше, чем в проекте "Ильинские луга".\n\nОтвет: "Восточное Бутово".'},
 

In [42]:
answers = [{'answer': el['answer'], 'answer_rag': el['answer_rag']} for el in questions]

In [44]:
metrics_prompt = ChatPromptTemplate.from_messages([
    ('system', prompts.sys_metrics_prompt_2),
    ('human', prompts.hum_metrics_prompt)
], template_format="jinja2")

In [45]:
metrics_message = metrics_prompt.format_messages(answers=str(answers))

In [46]:
metrics_response = llm.invoke(metrics_message)

In [47]:
text = metrics_response.content

In [48]:
data = ast.literal_eval(text)

In [49]:
metrics_df = pd.DataFrame(data)
metrics_df.head()

,id,correct
0,0,1
1,1,1
2,2,1
3,3,1
4,4,0


In [50]:
accuracy = metrics_df['correct'].sum() / len(metrics_df['correct'])
accuracy 

0.8